In [ ]:
!pip install spacy
!python -m spacy download pt_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 93.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import spacy
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from collections import Counter
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nlp = spacy.load('pt_core_news_sm')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
data = {'text': ['Nós adoramos Python!', 'Python é legal!', 'Ciência de dados é legal', None, 'Estou correndo para aprender Python.']}
df = pd.DataFrame(data)
df

,text
0,Nós adoramos Python!
1,Python é legal!
2,Ciência de dados é legal
3,None
4,Estou correndo para aprender Python.


In [ ]:
df['text'].fillna('', inplace=True)
df

/tmp/ipykernel_1136/1165055364.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['text'].fillna('', inplace=True)


,text
0,Nós adoramos Python!
1,Python é legal!
2,Ciência de dados é legal
3,
4,Estou correndo para aprender Python.


In [ ]:
def remove_noise(text):
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    return text

In [ ]:
df['cleaned_text'] = df['text'].apply(remove_noise)
df

,text,cleaned_text
0,Nós adoramos Python!,nós adoramos python
1,Python é legal!,python é legal
2,Ciência de dados é legal,ciência de dados é legal
3,,
4,Estou correndo para aprender Python.,estou correndo para aprender python


In [ ]:
df['tokenized_text'] = df['cleaned_text'].apply(nltk.word_tokenize)
df

,text,cleaned_text,tokenized_text
0,Nós adoramos Python!,nós adoramos python,"[nós, adoramos, python]"
1,Python é legal!,python é legal,"[python, é, legal]"
2,Ciência de dados é legal,ciência de dados é legal,"[ciência, de, dados, é, legal]"
3,,,[]
4,Estou correndo para aprender Python.,estou correndo para aprender python,"[estou, correndo, para, aprender, python]"


In [ ]:
stop_words = set(stopwords.words('portuguese'))
df['no_stopwords_text'] = df['tokenized_text'].apply(lambda x: [word for word in x if word not in stop_words])
df

,text,cleaned_text,tokenized_text,no_stopwords_text
0,Nós adoramos Python!,nós adoramos python,"[nós, adoramos, python]","[adoramos, python]"
1,Python é legal!,python é legal,"[python, é, legal]","[python, legal]"
2,Ciência de dados é legal,ciência de dados é legal,"[ciência, de, dados, é, legal]","[ciência, dados, legal]"
3,,,[],[]
4,Estou correndo para aprender Python.,estou correndo para aprender python,"[estou, correndo, para, aprender, python]","[correndo, aprender, python]"


In [ ]:
print(stop_words)

{'dele', 'elas', 'tem', 'uma', 'estavam', 'aquelas', 'fôramos', 'seriam', 'seríamos', 'houverem', 'essa', 'um', 'esteve', 'estava', 'depois', 'te', 'das', 'nas', 'aquela', 'somos', 'entre', 'houverei', 'teremos', 'tiveram', 'tenho', 'também', 'estamos', 'tenhamos', 'quem', 'esta', 'que', 'suas', 'teria', 'fora', 'vocês', 'nosso', 'sou', 'são', 'fossem', 'vos', 'mesmo', 'tua', 'hajam', 'por', 'nem', 'às', 'seremos', 'tenham', 'for', 'formos', 'era', 'qual', 'fui', 'eles', 'lhes', 'deles', 'lhe', 'ele', 'pela', 'nossas', 'à', 'isto', 'só', 'temos', 'houvéssemos', 'tenha', 'essas', 'tivéssemos', 'numa', 'ela', 'estivermos', 'haja', 'fôssemos', 'nossos', 'havemos', 'hei', 'ou', 'pelas', 'fosse', 'as', 'sejamos', 'aquilo', 'dela', 'aquele', 'você', 'da', 'estão', 'estivesse', 'houveriam', 'seus', 'serei', 'tém', 'tivemos', 'este', 'se', 'mais', 'quando', 'estive', 'terão', 'tivesse', 'tuas', 'meu', 'estivera', 'pelo', 'na', 'seja', 'tinha', 'ser', 'hão', 'esse', 'ao', 'foi', 'estejamos', 'f

In [ ]:
stemmer = PorterStemmer()
df['stemmed_text'] = df['no_stopwords_text'].apply(lambda x: [stemmer.stem(word) for word in x])

def lemmatize_text(text):
    doc = nlp(" ".join(text))
    return [token.lemma_ for token in doc]

df['lemmatized_text'] = df['no_stopwords_text'].apply(lemmatize_text)
df

,text,cleaned_text,tokenized_text,no_stopwords_text,stemmed_text,lemmatized_text
0,Nós adoramos Python!,nós adoramos python,"[nós, adoramos, python]","[adoramos, python]","[adoramo, python]","[adoramo, python]"
1,Python é legal!,python é legal,"[python, é, legal]","[python, legal]","[python, legal]","[python, legal]"
2,Ciência de dados é legal,ciência de dados é legal,"[ciência, de, dados, é, legal]","[ciência, dados, legal]","[ciência, dado, legal]","[ciência, dar, legal]"
3,,,[],[],[],[]
4,Estou correndo para aprender Python.,estou correndo para aprender python,"[estou, correndo, para, aprender, python]","[correndo, aprender, python]","[correndo, aprend, python]","[correr, aprender, python]"


In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['lemmatized_text'])
sequences = tokenizer.texts_to_sequences(df['lemmatized_text'])
one_hot_results = tokenizer.texts_to_matrix(df['lemmatized_text'], mode='binary')
one_hot_results

array([[0., 1., 0., 1., 0., 0., 0., 0.],
       [0., 1., 1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 1., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 1., 1.]])

In [ ]:
word_index = tokenizer.word_index
print("Word Index:", word_index)

Word Index: {'python': 1, 'legal': 2, 'adoramo': 3, 'ciência': 4, 'dar': 5, 'correr': 6, 'aprender': 7}


In [ ]:
max_len = 10
padded_sequences = pad_sequences(sequences, maxlen=max_len)
padded_sequences

array([[0, 0, 0, 0, 0, 0, 0, 0, 3, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 2],
       [0, 0, 0, 0, 0, 0, 0, 4, 5, 2],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 6, 7, 1]], dtype=int32)

In [ ]:
word_freq = Counter([word for sublist in df['lemmatized_text'] for word in sublist])
print("Palavras mais comuns:", word_freq.most_common(5))

Palavras mais comuns: [('python', 3), ('legal', 2), ('adoramo', 1), ('ciência', 1), ('dar', 1)]


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(df['lemmatized_text'], df['text'], test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print("Tamanho do conjunto de traino:", len(X_train))
print("Tamanho do cojunto de validação:", len(X_val))
print("Tamanho do conjunto de teste:", len(X_test))

Tamanho do conjunto de traino: 3
Tamanho do cojunto de validação: 1
Tamanho do conjunto de teste: 1
